In [ ]:
import math

from mesa import Model, Agent
from mesa.space import MultiGrid
import matplotlib.pyplot as plt

class Person(Agent):
    def __init__(self, uid, model):
        super().__init__(uid, model)

class City(Model):
    def __init__(self):
        super().__init__()
        self.grid = MultiGrid(10, 10, torus=False)


        self.persons = [Person(1, self), Person(2, self), Person(3, self)]

        for agent, pos in zip(self.persons, [(3, 3), (4, 3), (7, 8)]):
            self.grid.place_agent(agent, pos)

    def show(self):
        xs = [a.pos[0] for a in self.persons]
        ys = [a.pos[1] for a in self.persons]

        plt.grid()
        plt.scatter(xs, ys)
        plt.xlim(-0.5, 9.5)
        plt.ylim(-0.5, 9.5)
        plt.show()

model = City()
model.show()

print(list(model.grid.iter_neighbors((3, 3), moore=True)))

In [ ]:
from mesa import Model, Agent
from mesa.space import MultiGrid
import matplotlib.pyplot as plt
import random


class Person(Agent):
    def __init__(self, uid, model, color):
        super().__init__(uid, model)
        self.color = color


class City(Model):
    def __init__(self, width=10, height=10, num_agents=None):
        super().__init__()
        self.grid = MultiGrid(width, height, torus=False)

        # dit zorgt ervoor dat een random aantal agenten op het grid terecht komt.
        if num_agents is None:
            num_agents = random.randint(5, 15)

        self.persons = []
# alle agenten (dus husihoudens) krijgen een rode kleur op het grid
        possible_colors = ["red"]

        for i in range(num_agents):
            color = random.choice(possible_colors)
            person = Person(i, self, color)

            x = random.randint(0, width - 1)
            y = random.randint(0, height - 1)

            self.grid.place_agent(person, (x, y))
            self.persons.append(person)

    def show(self):
        plt.figure(figsize=(6, 6))

        for person in self.persons:
            x, y = person.pos
            plt.scatter(x, y, color=person.color, s=100)

        plt.xlim(-0.5, self.grid.width - 0.5)
        plt.ylim(-0.5, self.grid.height - 0.5)
        plt.xticks(range(self.grid.width))
        plt.yticks(range(self.grid.height))
        plt.grid(True)
        plt.show()


model = City()
model.show()

for person in model.persons:
    print(f"Agent {person.unique_id} staat op {person.pos} en heeft kleur {person.color}")

In [ ]:
from mesa import Model, Agent
from mesa.space import MultiGrid
import matplotlib.pyplot as plt
import random
import math

#dit is de eerste agent, de huishoudens. nu staan er nog geen variabelen in, maar dat is wat we later wel gaan toevoegen.
class Huishouden(Agent):
    def __init__(self, id, model):
        super().__init__(id, model)

#dit is de tweede agent, de containers. hier komen later ook nog meer karakteristieken bij
class Container(Agent):
    def __init__(self, id, model):
        super().__init__(id, model)

#hierbinnen gaan wij simuleren, deze karakteristieken moeten ook aangepast worden
class Stad(Model):
    def __init__(self, width=10, height=10, num_people=100, people_per_container=250):
        #width is de breedte, height de hoogte, num people aantal personen in de stad, people per container is de variabele die wij nog moeten beargumenteren
        super().__init__()

        self.grid = MultiGrid(width, height, torus=False)
        #multi grid houdt in dat er meerdere agents op een plek kunnen syaan, we moeten nadenken of dit wel kan met een huis? ik denk van wel omdat elk vakje een paar km is
        #door de torus stoppen we bij de rand en gaat helemaal rechts niet door links etc

        #lijst met alle huishoudens
        self.huishoudens = []
        #lijst met alle containers
        self.containers = []
        #telt alle unieke ids
        self.id = 0

        #deze formule bereknt het aantal containers aan de hand van de personen. de aantal containers worden altijd afgerond naar boven toe
        num_containers = math.ceil(num_people / people_per_container)

        #plaats de containers die berekend zijn in het grid
        for _ in range(num_containers):
            c = Container(self.next_id(), self)
            #dit geeft elke container een unieke plaats binnen de stad
            self.grid.place_agent(c, (random.randint(0, width-1), random.randint(0, height-1)))
            #voeg hem toe aan de lijst van de containers
            self.containers.append(c)

       #plaats de huishoudens in het grid/stad
        for _ in range(num_people):
            h = Huishouden(self.next_id(), self)
            self.grid.place_agent(h, (random.randint(0, width-1), random.randint(0, height-1)))
            self.huishoudens.append(h)

    #elke keer als er een agent gemaakt word gaat het id met 1 omhoog, zo voorkom je dat agents dezelfde id krijgen
    def next_id(self):
        self.id += 1
        return self.id

    def show(self):
        for p in self.huishoudens:
            plt.scatter(p.pos[0], p.pos[1], color="red", s=20)

        for c in self.containers:
            plt.scatter(c.pos[0], c.pos[1], color="green", s=100, marker="s")

        plt.grid()
        plt.xlim(-0.5, self.grid.width - 0.5)
        plt.ylim(-0.5, self.grid.height - 0.5)
        plt.show()


model = Stad(width=100, height=100, num_people=650, people_per_container=100)
model.show()

In [ ]:
from mesa import Model, Agent
from mesa.space import MultiGrid
import matplotlib.pyplot as plt
import random
import math


# Dit is de eerste agent: huishoudens
class Huishouden(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)


# Dit is de tweede agent: containers
class Container(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)


# Hierin simuleren we de stad
class Stad(Model):
    def __init__(self, width=10, height=10, num_people=5, people_per_container=10):
        super().__init__()

        # Grid waarin meerdere agents op dezelfde plek kunnen staan
        self.grid = MultiGrid(width, height, torus=False)

        # Lijsten voor huishoudens en containers
        self.huishoudens = []
        self.containers = []

        # Teller voor unieke ids
        self.id = 0

        # Bereken aantal containers
        num_containers = math.ceil(num_people / people_per_container)

        # Plaats de containers in het grid
        for _ in range(num_containers):
            c = Container(self.next_id(), self)
            self.grid.place_agent(c, (random.randint(0, width - 1), random.randint(0, height - 1)))
            self.containers.append(c)

        # Plaats de huishoudens in het grid
        for _ in range(num_people):
            h = Huishouden(self.next_id(), self)
            self.grid.place_agent(h, (random.randint(0, width - 1), random.randint(0, height - 1)))
            self.huishoudens.append(h)

    # Geeft elke agent een uniek id
    def next_id(self):
        self.id += 1
        return self.id

    # Bereken de Euclidische afstand tussen twee posities
    def euclidische_afstand(self, pos1, pos2):
        return math.sqrt((pos1[0] - pos2[0])**2 + (pos1[1] - pos2[1])**2)

    # Toon voor elk huishouden de afstand tot de container
    def toon_afstanden(self):
        container = self.containers[0]

        print(f"Container staat op positie: {container.pos}\n")

        for huishouden in self.huishoudens:
            afstand = self.euclidische_afstand(huishouden.pos, container.pos)
            print(f"Huishouden {huishouden.unique_id} staat op {huishouden.pos} --> afstand = {afstand:.2f}")

    # Laat het grid zien
    def show(self):
        for h in self.huishoudens:
            x, y = h.pos

            # Teken huishouden
            plt.scatter(x, y, color="red", s=40)

            # Label met unique_id
            plt.text(x, y, str(h.unique_id), fontsize=8, ha="center", va="bottom")

        for c in self.containers:
            x, y = c.pos

            # Teken container
            plt.scatter(x, y, color="green", s=120, marker="s")

            # Label voor container
            plt.text(x, y, "C", fontsize=10, ha="center", va="center", color="white")

        plt.grid()
        plt.xlim(-0.5, self.grid.width - 0.5)
        plt.ylim(-0.5, self.grid.height - 0.5)
        plt.show()


# Model maken en tonen
model = Stad(width=10, height=10, num_people=5, people_per_container=10)
model.show()
model.toon_afstanden()

In [ ]:
from mesa import Model, Agent
from mesa.space import MultiGrid
import matplotlib.pyplot as plt
import random
import math


# Agent: huishouden
class Huishouden(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)


# Agent: container
class Container(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)


# Model: stad
class Stad(Model):
    def __init__(self, width=10, height=10, num_people=5, people_per_container=2):
        super().__init__()

        self.grid = MultiGrid(width, height, torus=False)

        self.huishoudens = []
        self.containers = []

        self.id = 0

        # bereken aantal containers
        num_containers = math.ceil(num_people / people_per_container)

        # plaats containers
        for _ in range(num_containers):
            c = Container(self.next_id(), self)
            self.grid.place_agent(c, (random.randint(0, width - 1), random.randint(0, height - 1)))
            self.containers.append(c)

        # plaats huishoudens
        for _ in range(num_people):
            h = Huishouden(self.next_id(), self)
            self.grid.place_agent(h, (random.randint(0, width - 1), random.randint(0, height - 1)))
            self.huishoudens.append(h)

    # maak steeds een nieuw uniek id
    def next_id(self):
        self.id += 1
        return self.id

    # bereken euclidische afstand tussen twee posities
    def euclidische_afstand(self, pos1, pos2):
        return math.sqrt((pos1[0] - pos2[0])**2 + (pos1[1] - pos2[1])**2)

    # zoek voor een huishouden welke container het dichtstbij is
    def dichtstbijzijnde_container(self, huishouden):
        beste_container = None
        kleinste_afstand = float("inf")


        for container in self.containers:
            afstand = self.euclidische_afstand(huishouden.pos, container.pos)

            #dit omdat huishoudens altijd naar containers zullen lopen die het meest dichtbij zijn (aanname)
            if afstand < kleinste_afstand:
                kleinste_afstand = afstand
                beste_container = container

        return beste_container, kleinste_afstand

    # print per huishouden de dichtstbijzijnde container en de afstand
    def toon_resultaten(self):
        print("Containers:")
        for i, container in enumerate(self.containers, start=1):
            print(f"Container {i} staat op positie {container.pos}")

        print("\nHuishoudens:")
        for i, huishouden in enumerate(self.huishoudens, start=1):
            container, afstand = self.dichtstbijzijnde_container(huishouden)
            container_nummer = self.containers.index(container) + 1

            print(
                f"Huishouden {i} staat op {huishouden.pos} "
                f"--> gaat naar container {container_nummer} op {container.pos} "
                f"--> afstand = {afstand:.2f}"
            )

    # laat het grid zien
    def show(self):
        for i, h in enumerate(self.huishoudens, start=1):
            x, y = h.pos
            plt.scatter(x, y, color="red", s=40)
            plt.text(x, y, f"H{i}", fontsize=8, ha="center", va="bottom")

        for i, c in enumerate(self.containers, start=1):
            x, y = c.pos
            plt.scatter(x, y, color="green", s=120, marker="s")
            plt.text(x, y, f"C{i}", fontsize=8, ha="center", va="center", color="white")

        plt.grid()
        plt.xlim(-0.5, self.grid.width - 0.5)
        plt.ylim(-0.5, self.grid.height - 0.5)
        plt.show()


# model maken
model = Stad(width=10, height=10, num_people=5, people_per_container=2)
model.show()
model.toon_resultaten()

In [ ]:
# ik wil nu meenemen dat huishoudens buren kunnen hebben, voor het gemak rechts, links onder en boven. al deze buren geven een beinvloedingswaarde van +1. deze beinvloedingswaarde koppelt terug naar recyclegedrag. Ik denk dat recycle gedrag een random getal word tussen 0-1 en dat dan dus deze beinvloedingswaarde erbij komt. wanneer recycle gedrag boven een bepaald getal komt gaat een huishouden recycelen en anders niet. Voor nu neem ik aan dat elke buur zorgt voor +0.1 aan recyclegedarg erbij komt. dus 4 buren is +0.4


import random
import math
from mesa import Model, Agent
from mesa.space import MultiGrid
import matplotlib.pyplot as plt

# Agent: huishouden
class Huishouden(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)

        self.recyclewaarde = 0
        self.sociale_gevoeligheid = 0
        self.recyclet = False


# Agent: container
class Container(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)


# Model: stad
class Stad(Model):
    def __init__(self, width=10, height=10, num_people=5, people_per_container=2):
        super().__init__()

        self.grid = MultiGrid(width, height, torus=False)

        self.huishoudens = []
        self.containers = []

        self.id = 0

        # drempelwaarde
        self.drempel = 0.5

        # aantal containers berekenen
        num_containers = math.ceil(num_people / people_per_container)

        # containers plaatsen
        for _ in range(num_containers):
            c = Container(self.next_id(), self)
            self.grid.place_agent(c, (random.randint(0, width - 1), random.randint(0, height - 1)))
            self.containers.append(c)

        # huishoudens plaatsen
        for _ in range(num_people):
            h = Huishouden(self.next_id(), self)
            self.grid.place_agent(h, (random.randint(0, width - 1), random.randint(0, height - 1)))
            self.huishoudens.append(h)

        # recyclegedrag berekenen
        self.bereken_recyclegedrag()

    # uniek id
    def next_id(self):
        self.id += 1
        return self.id

    # afstand berekenen
    def euclidische_afstand(self, pos1, pos2):
        return math.sqrt((pos1[0] - pos2[0])**2 + (pos1[1] - pos2[1])**2)

    # dichtstbijzijnde container
    def dichtstbijzijnde_container(self, huishouden):
        beste_container = None
        kleinste_afstand = float("inf")

        for container in self.containers:
            afstand = self.euclidische_afstand(huishouden.pos, container.pos)

            if afstand < kleinste_afstand:
                kleinste_afstand = afstand
                beste_container = container

        return beste_container, kleinste_afstand

    # buren tellen (4 richtingen)
    def tel_buren(self, huishouden):
        x, y = huishouden.pos
        aantal_buren = 0

        buur_posities = [
            (x - 1, y),
            (x + 1, y),
            (x, y - 1),
            (x, y + 1)
        ]

        for bx, by in buur_posities:
            if 0 <= bx < self.grid.width and 0 <= by < self.grid.height:
                inhoud = self.grid.get_cell_list_contents([(bx, by)])

                for agent in inhoud:
                    if isinstance(agent, Huishouden):
                        aantal_buren += 1

        return aantal_buren

    # recyclegedrag bepalen
    def bereken_recyclegedrag(self):
        for huishouden in self.huishoudens:

            # random startwaarde
            huishouden.recyclewaarde = random.random()

            # buren tellen
            huishouden.sociale_gevoeligheid = self.tel_buren(huishouden)

            # +0.1 per buur
            huishouden.recyclewaarde += huishouden.sociale_gevoeligheid * 0.1

            # beslissing
            if huishouden.recyclewaarde > self.drempel:
                huishouden.recyclet = True
            else:
                huishouden.recyclet = False

    # nette tabel
    def toon_resultaten(self):
        print("\nRESULTATEN:\n")

        print(f"{'Huishouden':<12} {'Positie':<12} {'Buren':<6} {'Recyclewaarde':<16} {'Recyclet':<10}")
        print("-" * 60)

        for i, huishouden in enumerate(self.huishoudens, start=1):
            print(
                f"H{i:<11} "
                f"{str(huishouden.pos):<12} "
                f"{huishouden.sociale_gevoeligheid:<6} "
                f"{huishouden.recyclewaarde:<16.2f} "
                f"{str(huishouden.recyclet):<10}"
            )

    # visualisatie
    def show(self):
        for i, h in enumerate(self.huishoudens, start=1):
            x, y = h.pos

            if h.recyclet:
                kleur = "blue"
            else:
                kleur = "red"

            plt.scatter(x, y, color=kleur, s=40)
            plt.text(x, y, f"H{i}", fontsize=8, ha="center", va="bottom")

        for i, c in enumerate(self.containers, start=1):
            x, y = c.pos
            plt.scatter(x, y, color="green", s=120, marker="s")
            plt.text(x, y, f"C{i}", fontsize=8, ha="center", va="center", color="white")

        plt.grid()
        plt.xlim(-0.5, self.grid.width - 0.5)
        plt.ylim(-0.5, self.grid.height - 0.5)
        plt.show()


# model runnen
model = Stad(width=10, height=10, num_people=5, people_per_container=2)
model.show()
model.toon_resultaten()


#

In [ ]:
## versie 1
# ik wil nu meenemen dat huishoudens buren kunnen hebben, voor het gemak rechts, links onder en boven. al deze buren geven een beinvloedingswaarde van +1. deze beinvloedingswaarde koppelt terug naar recyclegedrag. Ik denk dat recycle gedrag een random getal word tussen 0-1 en dat dan dus deze beinvloedingswaarde erbij komt. wanneer recycle gedrag boven een bepaald getal komt gaat een huishouden recycelen en anders niet. Voor nu neem ik aan dat elke buur zorgt voor +0.1 aan recyclegedarg erbij komt. dus 4 buren is +0.4


import random
import math
from mesa import Model, Agent
from mesa.space import MultiGrid
import matplotlib.pyplot as plt


# Agent: huishouden
class Huishouden(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)
        self.recyclewaarde = 0
        self.sociale_gevoeligheid = 0
        self.recyclegedrag = False


# Agent: container
class Container(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)

# Model: stad
class Stad(Model):
    def __init__(self, width=10, height=10, num_people=5, people_per_container=2):
        super().__init__()

        self.grid = MultiGrid(width, height, torus=False)

        self.huishoudens = []
        self.containers = []

        self.id = 0

        # drempelwaarde
        self.drempel = 0.5

        # aantal containers berekenen
        num_containers = math.ceil(num_people / people_per_container)

        # containers plaatsen
        for _ in range(num_containers):
            c = Container(self.next_id(), self)
            self.grid.place_agent(c, (random.randint(0, width - 1), random.randint(0, height - 1)))
            self.containers.append(c)

        # huishoudens plaatsen
        for _ in range(num_people):
            h = Huishouden(self.next_id(), self)
            self.grid.place_agent(h, (random.randint(0, width - 1), random.randint(0, height - 1)))
            self.huishoudens.append(h)

        # recyclegedrag berekenen
        self.bereken_recyclegedrag()

    def next_id(self):
        self.id += 1
        return self.id

    # afstand berekenen
    def euclidische_afstand(self, pos1, pos2):
        return math.sqrt((pos1[0] - pos2[0]) ** 2 + (pos1[1] - pos2[1]) ** 2)

    # dichtstbijzijnde container
    def dichtstbijzijnde_container(self, huishouden):
        beste_container = None
        kleinste_afstand = float("inf")

        for container in self.containers:
            afstand = self.euclidische_afstand(huishouden.pos, container.pos)

            if afstand < kleinste_afstand:
                kleinste_afstand = afstand
                beste_container = container

        return beste_container, kleinste_afstand

    # buren tellen (4 richtingen)
    def tel_buren(self, huishouden):
        x, y = huishouden.pos
        aantal_buren = 0

        buur_posities = [
            (x - 1, y),
            (x + 1, y),
            (x, y - 1),
            (x, y + 1)
        ]

        for bx, by in buur_posities:
            if 0 <= bx < self.grid.width and 0 <= by < self.grid.height:
                inhoud = self.grid.get_cell_list_contents([(bx, by)])

                for agent in inhoud:
                    if isinstance(agent, Huishouden):
                        aantal_buren += 1

        return aantal_buren

    # recyclegedrag bepalen
    def bereken_recyclegedrag(self):
        for huishouden in self.huishoudens:
            # random startwaarde
            huishouden.recyclewaarde = random.random()
            # buren tellen
            huishouden.sociale_gevoeligheid = self.tel_buren(huishouden)
            # +0.1 per buur
            huishouden.recyclewaarde += huishouden.sociale_gevoeligheid * 0.1
            # -0.1 per afstandseenheid
            afstand = self.dichtstbijzijnde_container(huishouden)[1]
            huishouden.recyclewaarde -= afstand * 0.01
            # beslissing
            huishouden.recyclegedrag = huishouden.recyclewaarde > self.drempel

    # nette tabel
    def resultaten(self):
        print("\nRESULTATEN:\n")

        print(f"{'Huishouden':<12} {'Positie':<12} {'Buren':<6} {'Recyclewaarde':<16} {'Recyclet':<10}")
        print("-" * 60)

        for i, huishouden in enumerate(self.huishoudens, start=1):
            print(
                f"H{i:<11} "
                f"{str(huishouden.pos):<12} "
                f"{huishouden.sociale_gevoeligheid:<6} "
                f"{huishouden.recyclewaarde:<16.2f} "
                f"{str(huishouden.recyclegedrag):<10}"
            )

    # visualisatie
    def visualiseer(self):
        for i, h in enumerate(self.huishoudens, start=1):
            x, y = h.pos
            kleur = "blue" if h.recyclegedrag else "red"
            plt.scatter(x, y, color=kleur, s=60)
            plt.text(x, y, f"H{i}", fontsize=5, ha="center", va="center")

        for i, c in enumerate(self.containers, start=1):
            x, y = c.pos
            plt.scatter(x, y, color="green", s=100, marker="s")
            plt.text(x, y, f"C{i}", fontsize=6, ha="center", va="center")

        plt.grid()
        plt.title("Stad met huishoudens en textielcontainers")
        plt.xlim(-0.5, self.grid.width - 0.5)
        plt.ylim(-0.5, self.grid.height - 0.5)
        plt.show()

# model runnen
model = Stad(width=100, height=100, num_people=100, people_per_container=10)
model.visualiseer()
model.resultaten()

In [ ]:
## versie 2
# eigenschappen huishouden toegevoegd: opleidingsniveau, inkomen en geslacht
# hoeveelheid textielproductie per huishouden toegevoegd

import random
import math
from mesa import Model, Agent
from mesa.space import MultiGrid
import matplotlib.pyplot as plt


# Agent: huishouden
class Huishouden(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)
        self.recyclewaarde = 0
        self.sociale_gevoeligheid = 0
        self.recyclegedrag = False

        self.opleidingsniveau = random.choice(["laag", "midden", "hoog"])
        self.inkomen = random.choice(["laag", "midden", "hoog"])
        self.geslacht = random.choice(["man", "vrouw"])

        # Basis textielafval is 19 kg
        self.textielafval = 19

        # Pas de hoeveelheid textielafval aan op basis van inkomen
        if self.inkomen == "laag":
            self.textielafval *= 0.8  # Lager inkomen, minder textielafval
        elif self.inkomen == "hoog":
            self.textielafval *= 1.2  # Hoger inkomen, meer textielafval
        # "midden" inkomen hoeft geen aanpassing, dus blijft de basis van 19 kg

# Agent: container
class Container(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)

# Model: stad
class Stad(Model):
    def __init__(self, width=10, height=10, num_people=5, people_per_container=2):
        super().__init__()

        self.grid = MultiGrid(width, height, torus=False)

        self.huishoudens = []
        self.containers = []

        self.id = 0

        # drempelwaarde
        self.drempel = 0.5

        # aantal containers berekenen
        num_containers = math.ceil(num_people / people_per_container)

        # containers plaatsen
        for _ in range(num_containers):
            c = Container(self.next_id(), self)
            self.grid.place_agent(c, (random.randint(0, width - 1), random.randint(0, height - 1)))
            self.containers.append(c)

        # huishoudens plaatsen
        for _ in range(num_people):
            h = Huishouden(self.next_id(), self)
            self.grid.place_agent(h, (random.randint(0, width - 1), random.randint(0, height - 1)))
            self.huishoudens.append(h)

        # recyclegedrag berekenen
        self.bereken_recyclegedrag()

    def next_id(self):
        self.id += 1
        return self.id

    # afstand berekenen
    def euclidische_afstand(self, pos1, pos2):
        return math.sqrt((pos1[0] - pos2[0]) ** 2 + (pos1[1] - pos2[1]) ** 2)

    # dichtstbijzijnde container
    def dichtstbijzijnde_container(self, huishouden):
        beste_container = None
        kleinste_afstand = float("inf")

        for container in self.containers:
            afstand = self.euclidische_afstand(huishouden.pos, container.pos)

            if afstand < kleinste_afstand:
                kleinste_afstand = afstand
                beste_container = container

        return beste_container, kleinste_afstand

    # buren tellen (4 richtingen)
    def tel_buren(self, huishouden):
        x, y = huishouden.pos
        aantal_buren = 0

        buur_posities = [
            (x - 1, y),
            (x + 1, y),
            (x, y - 1),
            (x, y + 1)
        ]

        for bx, by in buur_posities:
            if 0 <= bx < self.grid.width and 0 <= by < self.grid.height:
                inhoud = self.grid.get_cell_list_contents([(bx, by)])

                for agent in inhoud:
                    if isinstance(agent, Huishouden):
                        aantal_buren += 1

        return aantal_buren

    # recyclegedrag bepalen
    def bereken_recyclegedrag(self):
        for huishouden in self.huishoudens:
            # random startwaarde
            huishouden.recyclewaarde = random.random()

            # buren tellen
            huishouden.sociale_gevoeligheid = self.tel_buren(huishouden)
            #afstand tot textielcontainer
            container, afstand = self.dichtstbijzijnde_container(huishouden)
            huishouden.afstand = afstand

            # +0.1 per buur
            huishouden.recyclewaarde += huishouden.sociale_gevoeligheid * 0.05
            # -0.1 per afstandseenheid tot textielcontainer
            huishouden.recyclewaarde -= afstand * 0.025
            # beslissing
            huishouden.recyclegedrag = huishouden.recyclewaarde > self.drempel

    # resultaten weergeven in nette tabel
    def resultaten(self):
        print("\nRESULTATEN:\n")

        print(f"{'Huishouden':<12} {'Positie':<12} {'Buren':<6} {'Opleiding':<10} {'Inkomen':<10} {'Geslacht':<10} {'Recyclewaarde':<16} {'Recyclegedrag':<10}")
        print("-" * 98)

        for i, huishouden in enumerate(self.huishoudens, start=1):
            print(
                f"H{i:<11} "
                f"{str(huishouden.pos):<12} "
                f"{huishouden.sociale_gevoeligheid:<6} "
                f"{huishouden.opleidingsniveau:<10} "
                f"{huishouden.inkomen:<10} "
                f"{huishouden.geslacht:<10} "
                f"{huishouden.afstand:<10.2f} "
                f"{huishouden.textielafval:<12.2f} "
                f"{huishouden.recyclewaarde:<16.2f} "
                f"{str(huishouden.recyclegedrag):<10}"
            )

    # visualisatie: stad met inwoners en textielcontainers
    def visualiseer(self):
        for i, h in enumerate(self.huishoudens, start=1):
            x, y = h.pos
            kleur = "blue" if h.recyclegedrag else "red"
            plt.scatter(x, y, color=kleur, s=60)
            plt.text(x, y, f"H{i}", fontsize=5, ha="center", va="center")

        for i, c in enumerate(self.containers, start=1):
            x, y = c.pos
            plt.scatter(x, y, color="green", s=100, marker="s")
            plt.text(x, y, f"C{i}", fontsize=6, ha="center", va="center")

        plt.grid()
        plt.title("Stad met huishoudens en textielcontainers")
        plt.xlim(-0.5, self.grid.width - 0.5)
        plt.ylim(-0.5, self.grid.height - 0.5)
        plt.show()

# model runnen
model = Stad(width=100, height=100, num_people=100, people_per_container=10)
model.visualiseer()
model.resultaten()

In [ ]:
## versie 3
# effecten van opleidingsniveau en geslacht meenemen in de recyclewaarde van een huishouden
# effect geslacht meenemen in hoeveelheid textielafval

import random
import math
from mesa import Model, Agent
from mesa.space import MultiGrid
import matplotlib.pyplot as plt


# Agent: huishouden
class Huishouden(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)
        self.recyclewaarde = 0
        self.sociale_gevoeligheid = 0
        self.recyclegedrag = False

        self.opleidingsniveau = random.choice(["laag", "midden", "hoog"])
        self.inkomen = random.choice(["laag", "midden", "hoog"])
        self.geslacht = random.choice(["man", "vrouw"])

        # Basis textielafval is 19 kg
        self.textielafval = 19

        # Correctie hoeveelheid textielafval obv. inkomen
        if self.inkomen == "laag":
            self.textielafval *= 0.8  #Lager inkomen, minder textielafval
        elif self.inkomen == "hoog":
            self.textielafval *= 1.2  #Hoger inkomen, meer textielafval
        #"midden" inkomen hoeft geen aanpassing, dus blijft 19 kg

        # Correctie hoeveelheid textielafval obv. geslacht
        if self.geslacht == "man":
            self.textielafval *= 0.9  # Mannen hebben minder textielafval
        elif self.geslacht == "vrouw":
            self.textielafval *= 1.1  # Vrouwen hebben meer textielafval

# Agent: container
class Container(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)

# Model: stad
class Stad(Model):
    def __init__(self, width=10, height=10, num_people=5, people_per_container=2):
        super().__init__()

        self.grid = MultiGrid(width, height, torus=False)

        self.huishoudens = []
        self.containers = []

        self.id = 0

        # drempelwaarde
        self.drempel = 0.5

        # aantal containers berekenen
        num_containers = math.ceil(num_people / people_per_container)

        # containers plaatsen
        for _ in range(num_containers):
            c = Container(self.next_id(), self)
            self.grid.place_agent(c, (random.randint(0, width - 1), random.randint(0, height - 1)))
            self.containers.append(c)

        # huishoudens plaatsen
        for _ in range(num_people):
            h = Huishouden(self.next_id(), self)
            self.grid.place_agent(h, (random.randint(0, width - 1), random.randint(0, height - 1)))
            self.huishoudens.append(h)

        # recyclegedrag berekenen
        self.bereken_recyclegedrag()

    def next_id(self):
        self.id += 1
        return self.id

    # afstand berekenen
    def euclidische_afstand(self, pos1, pos2):
        return math.sqrt((pos1[0] - pos2[0]) ** 2 + (pos1[1] - pos2[1]) ** 2)

    # dichtstbijzijnde container
    def dichtstbijzijnde_container(self, huishouden):
        beste_container = None
        kleinste_afstand = float("inf")

        for container in self.containers:
            afstand = self.euclidische_afstand(huishouden.pos, container.pos)

            if afstand < kleinste_afstand:
                kleinste_afstand = afstand
                beste_container = container

        return beste_container, kleinste_afstand

    # buren tellen (4 richtingen)
    def tel_buren(self, huishouden):
        x, y = huishouden.pos
        aantal_buren = 0

        buur_posities = [
            (x - 1, y),
            (x + 1, y),
            (x, y - 1),
            (x, y + 1)
        ]

        for bx, by in buur_posities:
            if 0 <= bx < self.grid.width and 0 <= by < self.grid.height:
                inhoud = self.grid.get_cell_list_contents([(bx, by)])

                for agent in inhoud:
                    if isinstance(agent, Huishouden):
                        aantal_buren += 1

        return aantal_buren

    # recyclegedrag bepalen
    def bereken_recyclegedrag(self):
        for huishouden in self.huishoudens:
            # random startwaarde
            huishouden.recyclewaarde = random.random()

            # buren tellen
            huishouden.sociale_gevoeligheid = self.tel_buren(huishouden)
            #afstand tot textielcontainer
            container, afstand = self.dichtstbijzijnde_container(huishouden)
            huishouden.afstand = afstand

            # +0.1 per buur
            huishouden.recyclewaarde += huishouden.sociale_gevoeligheid * 0.05
            # -0.1 per afstandseenheid tot textielcontainer
            huishouden.recyclewaarde -= afstand * 0.025

            # invloed inkomen op recyclewaarde (hoger inkomen -> meer milieubewust)
            if huishouden.inkomen == "hoog":
                huishouden.recyclewaarde += 0.1
            elif huishouden.inkomen == "laag":
                huishouden.recyclewaarde -= 0.1

            # invloed geslacht op recyclewaarde (vrouwen meer milieubewust dan mannen)
            if huishouden.geslacht == "vrouw":
                huishouden.recyclewaarde += 0.05
            elif huishouden.geslacht == "man":
                huishouden.recyclewaarde -= 0.05

            # beslissing
            huishouden.recyclegedrag = huishouden.recyclewaarde > self.drempel

    # resultaten weergeven in nette tabel
    def resultaten(self):
        print("\nRESULTATEN:\n")

        print(f"{'Huishouden':<12} {'Positie':<12} {'Buren':<6} {'Opleiding':<10} {'Inkomen':<10} {'Geslacht':<10} {'Afstand':<10} {'Textielafval':<12} {'Recyclewaarde':<16} {'Recyclegedrag':<10}")
        print("-" * 120)

        for i, huishouden in enumerate(self.huishoudens, start=1):
            print(
                f"H{i:<11} "
                f"{str(huishouden.pos):<12} "
                f"{huishouden.sociale_gevoeligheid:<6} "
                f"{huishouden.opleidingsniveau:<10} "
                f"{huishouden.inkomen:<10} "
                f"{huishouden.geslacht:<10} "
                f"{huishouden.afstand:<10.2f} "
                f"{huishouden.textielafval:<12.2f} "
                f"{huishouden.recyclewaarde:<16.2f} "
                f"{str(huishouden.recyclegedrag):<10}"
            )

    # visualisatie: stad met inwoners en textielcontainers
    def visualiseer(self):
        for i, h in enumerate(self.huishoudens, start=1):
            x, y = h.pos
            kleur = "blue" if h.recyclegedrag else "red"
            plt.scatter(x, y, color=kleur, s=60)
            plt.text(x, y, f"H{i}", fontsize=5, ha="center", va="center")

        for i, c in enumerate(self.containers, start=1):
            x, y = c.pos
            plt.scatter(x, y, color="green", s=100, marker="s")
            plt.text(x, y, f"C{i}", fontsize=6, ha="center", va="center")

        plt.grid()
        plt.title("Stad met huishoudens en textielcontainers")
        plt.xlim(-0.5, self.grid.width - 0.5)
        plt.ylim(-0.5, self.grid.height - 0.5)
        plt.show()

# model runnen
model = Stad(width=100, height=100, num_people=50, people_per_container=10)
model.visualiseer()
model.resultaten()

In [ ]:
## laatste versie (versie 4)
# eigenschappen textielcontainer toegevoegd: capaciteit, vulgraad, ledigingfrequentie
# stap functie toegevoegd (maar klopt nog niet helemaal)
# later nog toevoegen: textiel uberhaupt wel of niet weggooien (hoe vaak gooi je textiel weg?)
# later juiste waardes gebruiken voor invloed van bepaalde eigenschappen + juiste verhoudingen man/vrouw, inkomens, opleiding, aantal inwoners, aantal containers, etc.


import random
import math
from mesa import Model, Agent
from mesa.space import MultiGrid
import matplotlib.pyplot as plt


# Agent: huishouden
class Huishouden(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)
        self.recyclewaarde = 0
        self.sociale_gevoeligheid = 0
        self.recyclegedrag = False

        self.opleidingsniveau = random.choice(["laag", "midden", "hoog"])
        self.inkomen = random.choice(["laag", "midden", "hoog"])
        self.geslacht = random.choice(["man", "vrouw"])

        # Basis textielafval 12 kg/persoon/jaar
        self.textielafval = 12

        # Correctie hoeveelheid textielafval obv. inkomen
        if self.inkomen == "laag":
            self.textielafval *= 0.8  #Lager inkomen, minder textielafval
        elif self.inkomen == "hoog":
            self.textielafval *= 1.2  #Hoger inkomen, meer textielafval
        #"midden" inkomen hoeft geen aanpassing, dus blijft 12 kg/persoon/jaar

        # Correctie hoeveelheid textielafval obv. geslacht
        if self.geslacht == "man":
            self.textielafval *= 0.9  # Mannen hebben minder textielafval
        elif self.geslacht == "vrouw":
            self.textielafval *= 1.1  # Vrouwen hebben meer textielafval

# Agent: textielcontainer
class Container(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)


        self.capaciteit = 1250          # in kg (5 m³ * 250 kg/m³)
        self.vulgraad = 0               # Hoe vol is de container?
        self.ledigingsfrequentie = 8    # Lediging 1 keer per 8 weken
        self.week = 0                   # Huidige week in de simulatie

    # Container ledigen (reset de vulgraad naar 0)
    def ledigen(self):
        self.vulgraad = 0

# Model: stad
class Stad(Model):
    def __init__(self, width=10, height=10, num_people=5, people_per_container=2):
        super().__init__()

        self.grid = MultiGrid(width, height, torus=False)

        self.huishoudens = []
        self.containers = []

        self.id = 0

        # drempelwaarde
        self.drempel = 0.5

        # aantal containers berekenen
        num_containers = math.ceil(num_people / people_per_container)

        # containers plaatsen
        for _ in range(num_containers):
            c = Container(self.next_id(), self)
            self.grid.place_agent(c, (random.randint(0, width - 1), random.randint(0, height - 1)))
            self.containers.append(c)

        # huishoudens plaatsen
        for _ in range(num_people):
            h = Huishouden(self.next_id(), self)
            self.grid.place_agent(h, (random.randint(0, width - 1), random.randint(0, height - 1)))
            self.huishoudens.append(h)

        # recyclegedrag berekenen
        self.bereken_recyclegedrag()

    def next_id(self):
        self.id += 1
        return self.id

    # afstand berekenen
    def euclidische_afstand(self, pos1, pos2):
        return math.sqrt((pos1[0] - pos2[0]) ** 2 + (pos1[1] - pos2[1]) ** 2)

    # dichtstbijzijnde container
    def dichtstbijzijnde_container(self, huishouden):
        beste_container = None
        kleinste_afstand = float("inf")

        for container in self.containers:
            afstand = self.euclidische_afstand(huishouden.pos, container.pos)

            if afstand < kleinste_afstand:
                kleinste_afstand = afstand
                beste_container = container

        return beste_container, kleinste_afstand

    # buren tellen (4 richtingen)
    def tel_buren(self, huishouden):
        x, y = huishouden.pos
        aantal_buren = 0

        buur_posities = [
            (x - 1, y),
            (x + 1, y),
            (x, y - 1),
            (x, y + 1)
        ]

        for bx, by in buur_posities:
            if 0 <= bx < self.grid.width and 0 <= by < self.grid.height:
                inhoud = self.grid.get_cell_list_contents([(bx, by)])

                for agent in inhoud:
                    if isinstance(agent, Huishouden):
                        aantal_buren += 1

        return aantal_buren

    # recyclegedrag bepalen
    def bereken_recyclegedrag(self):
        for huishouden in self.huishoudens:
            # random startwaarde
            huishouden.recyclewaarde = random.random()

            # buren tellen
            huishouden.sociale_gevoeligheid = self.tel_buren(huishouden)
            #afstand tot textielcontainer
            container, afstand = self.dichtstbijzijnde_container(huishouden)
            huishouden.afstand = afstand

            # +0.1 per buur
            huishouden.recyclewaarde += huishouden.sociale_gevoeligheid * 0.05
            # -0.1 per afstandseenheid tot textielcontainer
            huishouden.recyclewaarde -= afstand * 0.025

            # invloed inkomen op recyclewaarde (hoger inkomen -> meer milieubewust)
            if huishouden.inkomen == "hoog":
                huishouden.recyclewaarde += 0.1
            elif huishouden.inkomen == "laag":
                huishouden.recyclewaarde -= 0.1

            # invloed geslacht op recyclewaarde (vrouwen meer milieubewust dan mannen)
            if huishouden.geslacht == "vrouw":
                huishouden.recyclewaarde += 0.05
            elif huishouden.geslacht == "man":
                huishouden.recyclewaarde -= 0.05

            # beslissing
            huishouden.recyclegedrag = huishouden.recyclewaarde > self.drempel

            # Huishouden recyclet het textielafval -> textielafval toevoegen aan vulgraad van de container
            if huishouden.recyclegedrag:
                # Geen overvolle container
                if container.vulgraad + huishouden.textielafval <= container.capaciteit:
                    container.vulgraad += huishouden.textielafval
                else:
                    # Als de container vol is, voeg alleen zo veel mogelijk textielafval toe en het resterende textielafval gaat bij het restafval
                    container.vulgraad = container.capaciteit

    # Simulatie van de tijd (om de containers te legen na elke week)
    def stap(self):
        # Simuleer het recyclegedrag van huishoudens
        self.bereken_recyclegedrag()

        # Update weeknummer in de simulatie
        for container in self.containers:
            container.week += 4  # Elke stap is 4 weken

        # Als het aantal weken een veelvoud van 8 is, leeg de container
        if container.week % 8 == 0:
            container.ledigen()  # Ledig de container

    # resultaten weergeven in nette tabel
    def resultaten(self):
        print("\nRESULTATEN:\n")

        print(f"{'Huishouden':<12} {'Positie':<12} {'Buren':<6} {'Opleiding':<10} {'Inkomen':<10} {'Geslacht':<10} {'Afstand':<10} {'Textielafval':<12} {'Recyclewaarde':<16} {'Recyclegedrag':<10}")
        print("-" * 120)

        for i, huishouden in enumerate(self.huishoudens, start=1):
            print(
                f"H{i:<11} "
                f"{str(huishouden.pos):<12} "
                f"{huishouden.sociale_gevoeligheid:<6} "
                f"{huishouden.opleidingsniveau:<10} "
                f"{huishouden.inkomen:<10} "
                f"{huishouden.geslacht:<10} "
                f"{huishouden.afstand:<10.2f} "
                f"{huishouden.textielafval:<12.2f} "
                f"{huishouden.recyclewaarde:<16.2f} "
                f"{str(huishouden.recyclegedrag):<10}"
            )

    # visualisatie: stad met inwoners en textielcontainers
    def visualiseer(self):
        for i, h in enumerate(self.huishoudens, start=1):
            x, y = h.pos
            kleur = "blue" if h.recyclegedrag else "red"
            plt.scatter(x, y, color=kleur, s=60)
            plt.text(x, y, f"H{i}", fontsize=5, ha="center", va="center")

        for i, c in enumerate(self.containers, start=1):
            x, y = c.pos
            plt.scatter(x, y, color="green", s=100, marker="s")
            plt.text(x, y, f"C{i}", fontsize=6, ha="center", va="center")

        plt.grid()
        plt.title("Stad met huishoudens en textielcontainers")
        plt.xlim(-0.5, self.grid.width - 0.5)
        plt.ylim(-0.5, self.grid.height - 0.5)
        plt.show()

# model
model = Stad(width=100, height=100, num_people=100, people_per_container=10)

# Simuleer 10 weken (4 stappen)
for week in range(4):
    print(f"\nWEEK {week + 1}")
    model.stap()
    model.visualiseer()
    model.resultaten()

In [ ]:
## laatste versie (versie 5)
# toegevoegd: huishoudens gooien textiel 1 keer per 20 weken weg, in de tussentijd sparen ze hun textiel op thuis
# later juiste waardes gebruiken voor invloed van bepaalde eigenschappen + juiste verhoudingen man/vrouw, inkomens, opleiding, aantal inwoners, aantal containers, etc.


import random
import math
from mesa import Model, Agent
from mesa.space import MultiGrid
import matplotlib.pyplot as plt


# Agent: huishouden
class Huishouden(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)
        self.recyclewaarde = 0
        self.sociale_gevoeligheid = 0
        self.recyclegedrag = False

        self.opleidingsniveau = random.choice(["laag", "midden", "hoog"])
        self.inkomen = random.choice(["laag", "midden", "hoog"])
        self.geslacht = random.choice(["man", "vrouw"])

        # Basis textielafval 12 kg/persoon/jaar
        self.textielafval_per_week = 0.23 * 2.10  # kg/huishouden/week
        self.textielafval_per_tijdstap = self.textielafval_per_week * 4
        self.opgespaard_textielafval = 0
        self.weggooifrequentie = 20  # in weken

        # Correctie hoeveelheid textielafval obv. inkomen
        if self.inkomen == "laag":
            self.textielafval_per_tijdstap *= 0.8  #Lager inkomen, minder textielafval
        elif self.inkomen == "hoog":
            self.textielafval_per_tijdstap *= 1.2  #Hoger inkomen, meer textielafval
        #"midden" inkomen hoeft geen aanpassing, dus blijft 12 kg/persoon/jaar

        # Correctie hoeveelheid textielafval obv. geslacht
        if self.geslacht == "man":
            self.textielafval_per_tijdstap *= 0.9  # Mannen hebben minder textielafval
        elif self.geslacht == "vrouw":
            self.textielafval_per_tijdstap *= 1.1  # Vrouwen hebben meer textielafval

# Agent: textielcontainer
class Container(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)


        self.capaciteit = 1250          # in kg (5 m³ * 250 kg/m³)
        self.vulgraad = 0               # Hoe vol is de container?
        self.ledigingsfrequentie = 8    # Lediging 1 keer per 8 weken
        self.week = 0                   # Huidige week in de simulatie

    # Container ledigen (reset de vulgraad naar 0)
    def ledigen(self):
        self.vulgraad = 0

# Model: stad
class Stad(Model):
    def __init__(self, width=10, height=10, num_people=5, people_per_container=2):
        super().__init__()

        self.grid = MultiGrid(width, height, torus=False)

        self.huishoudens = []
        self.containers = []

        self.id = 0
        self.week = 0

        # drempelwaarde
        self.drempel = 0.5

        # aantal containers berekenen
        num_containers = math.ceil(num_people / people_per_container)

        # containers plaatsen
        for _ in range(num_containers):
            c = Container(self.next_id(), self)
            self.grid.place_agent(c, (random.randint(0, width - 1), random.randint(0, height - 1)))
            self.containers.append(c)

        # huishoudens plaatsen
        for _ in range(num_people):
            h = Huishouden(self.next_id(), self)
            self.grid.place_agent(h, (random.randint(0, width - 1), random.randint(0, height - 1)))
            self.huishoudens.append(h)

        # recyclegedrag berekenen
        self.bereken_recyclegedrag()

    def next_id(self):
        self.id += 1
        return self.id

    # afstand berekenen
    def euclidische_afstand(self, pos1, pos2):
        return math.sqrt((pos1[0] - pos2[0]) ** 2 + (pos1[1] - pos2[1]) ** 2)

    # dichtstbijzijnde container
    def dichtstbijzijnde_container(self, huishouden):
        beste_container = None
        kleinste_afstand = float("inf")

        for container in self.containers:
            afstand = self.euclidische_afstand(huishouden.pos, container.pos)

            if afstand < kleinste_afstand:
                kleinste_afstand = afstand
                beste_container = container

        return beste_container, kleinste_afstand

    # buren tellen (4 richtingen)
    def tel_buren(self, huishouden):
        x, y = huishouden.pos
        aantal_buren = 0

        buur_posities = [
            (x - 1, y),
            (x + 1, y),
            (x, y - 1),
            (x, y + 1)
        ]

        for bx, by in buur_posities:
            if 0 <= bx < self.grid.width and 0 <= by < self.grid.height:
                inhoud = self.grid.get_cell_list_contents([(bx, by)])

                for agent in inhoud:
                    if isinstance(agent, Huishouden):
                        aantal_buren += 1

        return aantal_buren

    # recyclegedrag bepalen
    def bereken_recyclegedrag(self):
        for huishouden in self.huishoudens:
            # random startwaarde
            huishouden.recyclewaarde = random.random()

            # buren tellen
            huishouden.sociale_gevoeligheid = self.tel_buren(huishouden)
            #afstand tot textielcontainer
            container, afstand = self.dichtstbijzijnde_container(huishouden)
            huishouden.afstand = afstand

            # +0.1 per buur
            huishouden.recyclewaarde += huishouden.sociale_gevoeligheid * 0.05
            # -0.1 per afstandseenheid tot textielcontainer
            huishouden.recyclewaarde -= afstand * 0.025

            # invloed inkomen op recyclewaarde (hoger inkomen -> meer milieubewust)
            if huishouden.inkomen == "hoog":
                huishouden.recyclewaarde += 0.1
            elif huishouden.inkomen == "laag":
                huishouden.recyclewaarde -= 0.1

            # invloed geslacht op recyclewaarde (vrouwen meer milieubewust dan mannen)
            if huishouden.geslacht == "vrouw":
                huishouden.recyclewaarde += 0.05
            elif huishouden.geslacht == "man":
                huishouden.recyclewaarde -= 0.05

            # beslissing
            huishouden.recyclegedrag = huishouden.recyclewaarde > self.drempel

    # Simulatie van de tijd (om de containers te legen na elke week)
    def stap(self):
        self.week += 4 #weken

        # huishoudens produceren textielafval en sparen dat op
        for huishouden in self.huishoudens:
            huishouden.opgespaard_textielafval += huishouden.textielafval_per_tijdstap

        # recyclegedrag opnieuw berekenen
        self.bereken_recyclegedrag()

        # huishoudens gooien gemiddeld 1 keer per 20 weken textiel weg
        if self.week % 20 == 0:
            for huishouden in self.huishoudens:
                if huishouden.recyclegedrag:
                    container, afstand = self.dichtstbijzijnde_container(huishouden)

                    if container.vulgraad + huishouden.opgespaard_textielafval <= container.capaciteit:
                        container.vulgraad += huishouden.opgespaard_textielafval
                    else:
                        container.vulgraad = container.capaciteit

                # na weggooimoment is de voorraad thuis weg
                huishouden.opgespaard_textielafval = 0

        # containers worden 1 keer per 8 weken geleegd
        if self.week % 8 == 0:
            for container in self.containers:
                container.ledigen()

    # resultaten weergeven in nette tabel
    def resultaten(self):
        print("\nRESULTATEN:\n")

        print(f"{'Huishouden':<12} {'Positie':<12} {'Buren':<6} {'Opleiding':<10} {'Inkomen':<10} {'Geslacht':<10} {'Afstand':<10} {'Opgespaard':<12} {'Recyclewaarde':<16} {'Recyclegedrag':<10}")
        print("-" * 120)

        for i, huishouden in enumerate(self.huishoudens, start=1):
            print(
                f"H{i:<11} "
                f"{str(huishouden.pos):<12} "
                f"{huishouden.sociale_gevoeligheid:<6} "
                f"{huishouden.opleidingsniveau:<10} "
                f"{huishouden.inkomen:<10} "
                f"{huishouden.geslacht:<10} "
                f"{huishouden.afstand:<10.2f} "
                f"{huishouden.opgespaard_textielafval:<12.2f} "
                f"{huishouden.recyclewaarde:<16.2f} "
                f"{str(huishouden.recyclegedrag):<10}"
            )

    # visualisatie: stad met inwoners en textielcontainers
    def visualiseer(self):
        for i, h in enumerate(self.huishoudens, start=1):
            x, y = h.pos
            kleur = "blue" if h.recyclegedrag else "red"
            plt.scatter(x, y, color=kleur, s=60)
            plt.text(x, y, f"H{i}", fontsize=5, ha="center", va="center")

        for i, c in enumerate(self.containers, start=1):
            x, y = c.pos
            plt.scatter(x, y, color="green", s=100, marker="s")
            plt.text(x, y, f"C{i}", fontsize=6, ha="center", va="center")

        plt.grid()
        plt.title("Stad met huishoudens en textielcontainers")
        plt.xlim(-0.5, self.grid.width - 0.5)
        plt.ylim(-0.5, self.grid.height - 0.5)
        plt.show()

# model
model = Stad(width=100, height=100, num_people=100, people_per_container=10)

# Simuleer 10 weken (4 stappen)
for week in range(4):
    print(f"\nWEEK {week + 1}")
    model.stap()
    model.visualiseer()
    model.resultaten()

In [ ]:
import random
import math
from mesa import Model, Agent
from mesa.space import MultiGrid
import matplotlib.pyplot as plt


# Agent: huishouden
class Huishouden(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)
        self.recyclewaarde = 0
        self.sociale_gevoeligheid = 0
        self.recyclegedrag = False

        self.opleidingsniveau = random.choice(["laag", "midden", "hoog"])
        self.inkomen = random.choice(["laag", "midden", "hoog"])
        self.geslacht = random.choice(["man", "vrouw"])

        # Basis textielafval 12 kg/persoon/jaar
        self.textielafval_per_week = 0.23 * 2.10  # kg/huishouden/week
        self.textielafval_per_tijdstap = self.textielafval_per_week * 4
        self.opgespaard_textielafval = 0
        self.weggooifrequentie = 20  # in weken

        # Correctie hoeveelheid textielafval obv. inkomen
        if self.inkomen == "laag":
            self.textielafval_per_tijdstap *= 0.8  # Lager inkomen, minder textielafval
        elif self.inkomen == "hoog":
            self.textielafval_per_tijdstap *= 1.2  # Hoger inkomen, meer textielafval

        # Correctie hoeveelheid textielafval obv. geslacht
        if self.geslacht == "man":
            self.textielafval_per_tijdstap *= 0.9  # Mannen hebben minder textielafval
        elif self.geslacht == "vrouw":
            self.textielafval_per_tijdstap *= 1.1  # Vrouwen hebben meer textielafval


# Agent: textielcontainer
class Container(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)

        self.capaciteit = 1250          # in kg (5 m³ * 250 kg/m³)
        self.vulgraad = 0               # Hoe vol is de container?
        self.ledigingsfrequentie = 8    # Lediging 1 keer per 8 weken
        self.week = 0                   # Huidige week in de simulatie

    # Container ledigen (reset de vulgraad naar 0)
    def ledigen(self):
        self.vulgraad = 0


# Model: stad
class Stad(Model):
    def __init__(self, width=10, height=10, num_people=5, people_per_container=2):
        super().__init__()

        self.grid = MultiGrid(width, height, torus=False)

        self.huishoudens = []
        self.containers = []

        self.id = 0
        self.week = 0

        # drempelwaarde
        self.drempel = 0.5

        # aantal containers berekenen
        num_containers = math.ceil(num_people / people_per_container)

        # lijst met alle gehele coördinaten op het grid
        beschikbare_posities = [(x, y) for x in range(width) for y in range(height)]
        random.shuffle(beschikbare_posities)

        # containers plaatsen op unieke gehele coördinaten
        for _ in range(num_containers):
            c = Container(self.next_id(), self)
            pos = beschikbare_posities.pop()
            self.grid.place_agent(c, pos)
            self.containers.append(c)

        # huishoudens plaatsen op unieke gehele coördinaten
        for _ in range(num_people):
            h = Huishouden(self.next_id(), self)
            pos = beschikbare_posities.pop()
            self.grid.place_agent(h, pos)
            self.huishoudens.append(h)

        # recyclegedrag berekenen
        self.bereken_recyclegedrag()

    def next_id(self):
        self.id += 1
        return self.id

    # Manhattan-afstand berekenen op het grid
    def manhattan_afstand(self, pos1, pos2):
        return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])

    # dichtstbijzijnde container
    def dichtstbijzijnde_container(self, huishouden):
        beste_container = None
        kleinste_afstand = float("inf")

        for container in self.containers:
            afstand = self.manhattan_afstand(huishouden.pos, container.pos)

            if afstand < kleinste_afstand:
                kleinste_afstand = afstand
                beste_container = container

        return beste_container, kleinste_afstand

    # buren tellen (alleen boven, onder, links, rechts)
    def tel_buren(self, huishouden):
        x, y = huishouden.pos
        aantal_buren = 0

        buur_posities = [
            (x - 1, y),
            (x + 1, y),
            (x, y - 1),
            (x, y + 1)
        ]

        for bx, by in buur_posities:
            if 0 <= bx < self.grid.width and 0 <= by < self.grid.height:
                inhoud = self.grid.get_cell_list_contents([(bx, by)])

                for agent in inhoud:
                    if isinstance(agent, Huishouden):
                        aantal_buren += 1

        return aantal_buren

    # recyclegedrag bepalen
    def bereken_recyclegedrag(self):
        for huishouden in self.huishoudens:
            # random startwaarde
            huishouden.recyclewaarde = random.random()

            # buren tellen
            huishouden.sociale_gevoeligheid = self.tel_buren(huishouden)

            # afstand tot textielcontainer
            container, afstand = self.dichtstbijzijnde_container(huishouden)
            huishouden.afstand = afstand

            # +0.05 per buur
            huishouden.recyclewaarde += huishouden.sociale_gevoeligheid * 0.05

            # -0.025 per afstandseenheid tot textielcontainer
            huishouden.recyclewaarde -= afstand * 0.025

            # invloed inkomen op recyclewaarde
            if huishouden.inkomen == "hoog":
                huishouden.recyclewaarde += 0.1
            elif huishouden.inkomen == "laag":
                huishouden.recyclewaarde -= 0.1

            # invloed geslacht op recyclewaarde
            if huishouden.geslacht == "vrouw":
                huishouden.recyclewaarde += 0.05
            elif huishouden.geslacht == "man":
                huishouden.recyclewaarde -= 0.05

            # beslissing
            huishouden.recyclegedrag = huishouden.recyclewaarde > self.drempel

    # Simulatie van de tijd
    def stap(self):
        self.week += 4  # weken

        # huishoudens produceren textielafval en sparen dat op
        for huishouden in self.huishoudens:
            huishouden.opgespaard_textielafval += huishouden.textielafval_per_tijdstap

        # recyclegedrag opnieuw berekenen
        self.bereken_recyclegedrag()

        # huishoudens gooien gemiddeld 1 keer per 20 weken textiel weg
        if self.week % 20 == 0:
            for huishouden in self.huishoudens:
                if huishouden.recyclegedrag:
                    container, afstand = self.dichtstbijzijnde_container(huishouden)

                    if container.vulgraad + huishouden.opgespaard_textielafval <= container.capaciteit:
                        container.vulgraad += huishouden.opgespaard_textielafval
                    else:
                        container.vulgraad = container.capaciteit

                # na weggooimoment is de voorraad thuis weg
                huishouden.opgespaard_textielafval = 0

        # containers worden 1 keer per 8 weken geleegd
        if self.week % 8 == 0:
            for container in self.containers:
                container.ledigen()

    # resultaten weergeven in nette tabel
    def resultaten(self):
        print("\nRESULTATEN:\n")

        print(f"{'Huishouden':<12} {'Positie':<12} {'Buren':<6} {'Opleiding':<10} {'Inkomen':<10} {'Geslacht':<10} {'Afstand':<10} {'Opgespaard':<12} {'Recyclewaarde':<16} {'Recyclegedrag':<10}")
        print("-" * 120)

        for i, huishouden in enumerate(self.huishoudens, start=1):
            print(
                f"H{i:<11} "
                f"{str(huishouden.pos):<12} "
                f"{huishouden.sociale_gevoeligheid:<6} "
                f"{huishouden.opleidingsniveau:<10} "
                f"{huishouden.inkomen:<10} "
                f"{huishouden.geslacht:<10} "
                f"{huishouden.afstand:<10.2f} "
                f"{huishouden.opgespaard_textielafval:<12.2f} "
                f"{huishouden.recyclewaarde:<16.2f} "
                f"{str(huishouden.recyclegedrag):<10}"
            )

    # visualisatie: stad met inwoners en textielcontainers
    def visualiseer(self):
        for i, h in enumerate(self.huishoudens, start=1):
            x, y = h.pos
            kleur = "blue" if h.recyclegedrag else "red"
            plt.scatter(x, y, color=kleur, s=60)
            plt.text(x, y, f"H{i}", fontsize=5, ha="center", va="center")

        for i, c in enumerate(self.containers, start=1):
            x, y = c.pos
            plt.scatter(x, y, color="green", s=100, marker="s")
            plt.text(x, y, f"C{i}", fontsize=6, ha="center", va="center")

        plt.grid()
        plt.title("Stad met huishoudens en textielcontainers")
        plt.xlim(-0.5, self.grid.width - 0.5)
        plt.ylim(-0.5, self.grid.height - 0.5)
        plt.show()


# model
model = Stad(width=100, height=100, num_people=100, people_per_container=10)

# Simuleer 10 weken (4 stappen)
for week in range(4):
    print(f"\nWEEK {week + 1}")
    model.stap()
    model.visualiseer()
    model.resultaten()

In [ ]:
#ik ga nu even het grid wat kleiner maken om te checken of ze wel buren hebben meer dan 1
#ook ga ik instellen dat als een huishouden naast een container woont je altijd recyclet, door dan in te stellen dat je recycel waarde altijd boven het minimum uitkomt
import random
import math
from mesa import Model, Agent
from mesa.space import MultiGrid
import matplotlib.pyplot as plt


# Agent: huishouden
class Huishouden(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)
        self.recyclewaarde = 0
        self.sociale_gevoeligheid = 0
        self.recyclegedrag = False

        self.opleidingsniveau = random.choice(["laag", "midden", "hoog"])
        self.inkomen = random.choice(["laag", "midden", "hoog"])
        self.geslacht = random.choice(["man", "vrouw"])

        # Basis textielafval 12 kg/persoon/jaar
        self.textielafval_per_week = 0.23 * 2.10  # kg/huishouden/week
        self.textielafval_per_tijdstap = self.textielafval_per_week * 4
        self.opgespaard_textielafval = 0
        self.weggooifrequentie = 20  # in weken

        # Correctie hoeveelheid textielafval obv. inkomen
        if self.inkomen == "laag":
            self.textielafval_per_tijdstap *= 0.8
        elif self.inkomen == "hoog":
            self.textielafval_per_tijdstap *= 1.2

        # Correctie hoeveelheid textielafval obv. geslacht
        if self.geslacht == "man":
            self.textielafval_per_tijdstap *= 0.9
        elif self.geslacht == "vrouw":
            self.textielafval_per_tijdstap *= 1.1


# Agent: textielcontainer
class Container(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)

        self.capaciteit = 1250          # in kg (5 m³ * 250 kg/m³)
        self.vulgraad = 0               # Hoe vol is de container?
        self.ledigingsfrequentie = 8    # Lediging 1 keer per 8 weken
        self.week = 0                   # Huidige week in de simulatie

    # Container ledigen (reset de vulgraad naar 0)
    def ledigen(self):
        self.vulgraad = 0


# Model: stad
class Stad(Model):
    def __init__(self, width=10, height=10, num_people=5, people_per_container=2):
        super().__init__()

        self.grid = MultiGrid(width, height, torus=False)

        self.huishoudens = []
        self.containers = []

        self.id = 0
        self.week = 0

        # drempelwaarde
        self.drempel = 0.5

        # aantal containers berekenen
        num_containers = math.ceil(num_people / people_per_container)

        # lijst met alle gehele coördinaten op het grid
        beschikbare_posities = [(x, y) for x in range(width) for y in range(height)]
        random.shuffle(beschikbare_posities)

        # containers plaatsen op unieke gehele coördinaten
        for _ in range(num_containers):
            c = Container(self.next_id(), self)
            pos = beschikbare_posities.pop()
            self.grid.place_agent(c, pos)
            self.containers.append(c)

        # huishoudens plaatsen op unieke gehele coördinaten
        for _ in range(num_people):
            h = Huishouden(self.next_id(), self)
            pos = beschikbare_posities.pop()
            self.grid.place_agent(h, pos)
            self.huishoudens.append(h)

        # recyclegedrag berekenen
        self.bereken_recyclegedrag()

    def next_id(self):
        self.id += 1
        return self.id

    # Manhattan-afstand berekenen op het grid
    def manhattan_afstand(self, pos1, pos2):
        return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])

    # dichtstbijzijnde container
    def dichtstbijzijnde_container(self, huishouden):
        beste_container = None
        kleinste_afstand = float("inf")

        for container in self.containers:
            afstand = self.manhattan_afstand(huishouden.pos, container.pos)

            if afstand < kleinste_afstand:
                kleinste_afstand = afstand
                beste_container = container

        return beste_container, kleinste_afstand

    # buren tellen (alleen boven, onder, links, rechts)
    def tel_buren(self, huishouden):
        x, y = huishouden.pos
        aantal_buren = 0

        buur_posities = [
            (x - 1, y),
            (x + 1, y),
            (x, y - 1),
            (x, y + 1)
        ]

        for bx, by in buur_posities:
            if 0 <= bx < self.grid.width and 0 <= by < self.grid.height:
                inhoud = self.grid.get_cell_list_contents([(bx, by)])

                for agent in inhoud:
                    if isinstance(agent, Huishouden):
                        aantal_buren += 1

        return aantal_buren

    # recyclegedrag bepalen
    def bereken_recyclegedrag(self):
        for huishouden in self.huishoudens:
            # random startwaarde
            huishouden.recyclewaarde = random.random()

            # buren tellen
            huishouden.sociale_gevoeligheid = self.tel_buren(huishouden)

            # afstand tot textielcontainer
            container, afstand = self.dichtstbijzijnde_container(huishouden)
            huishouden.afstand = afstand

            # ALS huishouden direct naast container woont -> altijd recyclen
            if afstand == 1:
                huishouden.recyclewaarde = self.drempel + 1
                huishouden.recyclegedrag = True
                continue

            # +0.05 per buur
            huishouden.recyclewaarde += huishouden.sociale_gevoeligheid * 0.05

            # -0.025 per afstandseenheid tot textielcontainer
            huishouden.recyclewaarde -= afstand * 0.025

            # invloed inkomen op recyclewaarde
            if huishouden.inkomen == "hoog":
                huishouden.recyclewaarde += 0.1
            elif huishouden.inkomen == "laag":
                huishouden.recyclewaarde -= 0.1

            # invloed geslacht op recyclewaarde
            if huishouden.geslacht == "vrouw":
                huishouden.recyclewaarde += 0.05
            elif huishouden.geslacht == "man":
                huishouden.recyclewaarde -= 0.05

            # beslissing
            huishouden.recyclegedrag = huishouden.recyclewaarde > self.drempel

    # Simulatie van de tijd
    def stap(self):
        self.week += 4  # weken

        # huishoudens produceren textielafval en sparen dat op
        for huishouden in self.huishoudens:
            huishouden.opgespaard_textielafval += huishouden.textielafval_per_tijdstap

        # recyclegedrag opnieuw berekenen
        self.bereken_recyclegedrag()

        # huishoudens gooien gemiddeld 1 keer per 20 weken textiel weg
        if self.week % 20 == 0:
            for huishouden in self.huishoudens:
                if huishouden.recyclegedrag:
                    container, afstand = self.dichtstbijzijnde_container(huishouden)

                    if container.vulgraad + huishouden.opgespaard_textielafval <= container.capaciteit:
                        container.vulgraad += huishouden.opgespaard_textielafval
                    else:
                        container.vulgraad = container.capaciteit

                # na weggooimoment is de voorraad thuis weg
                huishouden.opgespaard_textielafval = 0

        # containers worden 1 keer per 8 weken geleegd
        if self.week % 8 == 0:
            for container in self.containers:
                container.ledigen()

    # resultaten weergeven in nette tabel
    def resultaten(self):
        print("\nRESULTATEN:\n")

        print(f"{'Huishouden':<12} {'Positie':<12} {'Buren':<6} {'Opleiding':<10} {'Inkomen':<10} {'Geslacht':<10} {'Afstand':<10} {'Opgespaard':<12} {'Recyclewaarde':<16} {'Recyclegedrag':<10}")
        print("-" * 120)

        for i, huishouden in enumerate(self.huishoudens, start=1):
            print(
                f"H{i:<11} "
                f"{str(huishouden.pos):<12} "
                f"{huishouden.sociale_gevoeligheid:<6} "
                f"{huishouden.opleidingsniveau:<10} "
                f"{huishouden.inkomen:<10} "
                f"{huishouden.geslacht:<10} "
                f"{huishouden.afstand:<10.2f} "
                f"{huishouden.opgespaard_textielafval:<12.2f} "
                f"{huishouden.recyclewaarde:<16.2f} "
                f"{str(huishouden.recyclegedrag):<10}"
            )

    # visualisatie: stad met inwoners en textielcontainers
    def visualiseer(self):
        for i, h in enumerate(self.huishoudens, start=1):
            x, y = h.pos
            kleur = "blue" if h.recyclegedrag else "red"
            plt.scatter(x, y, color=kleur, s=60)
            plt.text(x, y, f"H{i}", fontsize=5, ha="center", va="center")

        for i, c in enumerate(self.containers, start=1):
            x, y = c.pos
            plt.scatter(x, y, color="green", s=100, marker="s")
            plt.text(x, y, f"C{i}", fontsize=6, ha="center", va="center")

        plt.grid()
        plt.title("Stad met huishoudens en textielcontainers")
        plt.xlim(-0.5, self.grid.width - 0.5)
        plt.ylim(-0.5, self.grid.height - 0.5)
        plt.show()


# model
model = Stad(width=10, height=10, num_people=80, people_per_container=10)

# Simuleer 10 weken (4 stappen)
for week in range(4):
    print(f"\nWEEK {week + 1}")
    model.stap()
    model.visualiseer()
    model.resultaten()

In [ ]:
# we kijken nu ook naar wat een buur doet. als een buur recyecelt zal je eigen recycel waarde 0.1 srijgen, maar het kan ook -0.1 gaan als de buur dus juist niet recycelt
#nieuwste versie
import random
import math
from mesa import Model, Agent
from mesa.space import MultiGrid
import matplotlib.pyplot as plt


# Agent: huishouden
class Huishouden(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)
        self.recyclewaarde = 0
        self.sociale_gevoeligheid = 0
        self.recyclegedrag = False

        self.opleidingsniveau = random.choice(["laag", "midden", "hoog"])
        self.inkomen = random.choice(["laag", "midden", "hoog"])
        self.geslacht = random.choice(["man", "vrouw"])

        # Basis textielafval 12 kg/persoon/jaar
        self.textielafval_per_week = 0.23 * 2.10  # kg/huishouden/week
        self.textielafval_per_tijdstap = self.textielafval_per_week * 4
        self.opgespaard_textielafval = 0
        self.weggooifrequentie = 20  # in weken

        # Correctie hoeveelheid textielafval obv inkomen
        if self.inkomen == "laag":
            self.textielafval_per_tijdstap *= 0.8
        elif self.inkomen == "hoog":
            self.textielafval_per_tijdstap *= 1.2

        # Correctie hoeveelheid textielafval obv geslacht
        if self.geslacht == "man":
            self.textielafval_per_tijdstap *= 0.9
        elif self.geslacht == "vrouw":
            self.textielafval_per_tijdstap *= 1.1


# Agent: textielcontainer
class Container(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)

        self.capaciteit = 1250          # in kg (5 m³ * 250 kg/m³)
        self.vulgraad = 0
        self.ledigingsfrequentie = 8    # 1 keer per 8 weken
        self.week = 0

    def ledigen(self):
        self.vulgraad = 0


# Model: stad
class Stad(Model):
    def __init__(self, width=10, height=10, num_people=90, people_per_container=10):
        super().__init__()

        self.grid = MultiGrid(width, height, torus=False)

        self.huishoudens = []
        self.containers = []

        self.id = 0
        self.week = 0
        self.drempel = 0.5

        # aantal containers berekenen
        num_containers = math.ceil(num_people / people_per_container)

        # check of alles op het grid past
        totaal_agents = num_people + num_containers
        totaal_cellen = width * height
        if totaal_agents > totaal_cellen:
            raise ValueError(
                f"Te veel agents voor het grid: {totaal_agents} agents voor {totaal_cellen} cellen."
            )

        # alle gehele coördinaten op het grid
        beschikbare_posities = [(x, y) for x in range(width) for y in range(height)]
        random.shuffle(beschikbare_posities)

        # containers plaatsen
        for _ in range(num_containers):
            c = Container(self.next_id(), self)
            pos = beschikbare_posities.pop()
            self.grid.place_agent(c, pos)
            self.containers.append(c)

        # huishoudens plaatsen
        for _ in range(num_people):
            h = Huishouden(self.next_id(), self)
            pos = beschikbare_posities.pop()
            self.grid.place_agent(h, pos)
            self.huishoudens.append(h)

        # eerste berekening
        self.bereken_recyclegedrag()

    def next_id(self):
        self.id += 1
        return self.id

    # Manhattan-afstand op het grid
    def manhattan_afstand(self, pos1, pos2):
        return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])

    # dichtstbijzijnde container
    def dichtstbijzijnde_container(self, huishouden):
        beste_container = None
        kleinste_afstand = float("inf")

        for container in self.containers:
            afstand = self.manhattan_afstand(huishouden.pos, container.pos)

            if afstand < kleinste_afstand:
                kleinste_afstand = afstand
                beste_container = container

        return beste_container, kleinste_afstand

    # buurinvloed berekenen:
    # +0.1 voor buur die recycelt
    # -0.1 voor buur die niet recycelt
    def bereken_buurinvloed(self, huishouden):
        x, y = huishouden.pos
        aantal_buren = 0
        buurinvloed = 0

        buur_posities = [
            (x - 1, y),  # links
            (x + 1, y),  # rechts
            (x, y - 1),  # onder
            (x, y + 1)   # boven
        ]

        for bx, by in buur_posities:
            if 0 <= bx < self.grid.width and 0 <= by < self.grid.height:
                inhoud = self.grid.get_cell_list_contents([(bx, by)])

                for agent in inhoud:
                    if isinstance(agent, Huishouden):
                        aantal_buren += 1
                        if agent.recyclegedrag:
                            buurinvloed += 0.1
                        else:
                            buurinvloed -= 0.1

        return aantal_buren, buurinvloed

    # recyclegedrag bepalen
    def bereken_recyclegedrag(self):
        for huishouden in self.huishoudens:
            # random startwaarde
            huishouden.recyclewaarde = random.random()

            # buurinvloed
            aantal_buren, buurinvloed = self.bereken_buurinvloed(huishouden)
            huishouden.sociale_gevoeligheid = aantal_buren

            # afstand tot dichtstbijzijnde container
            container, afstand = self.dichtstbijzijnde_container(huishouden)
            huishouden.afstand = afstand

            # als huishouden direct naast een container woont -> altijd recyclen
            if afstand == 1:
                huishouden.recyclewaarde = self.drempel + 1
                huishouden.recyclegedrag = True
                continue

            # invloed buren
            huishouden.recyclewaarde += buurinvloed

            # invloed afstand
            huishouden.recyclewaarde -= afstand * 0.025

            # invloed inkomen
            if huishouden.inkomen == "hoog":
                huishouden.recyclewaarde += 0.1
            elif huishouden.inkomen == "laag":
                huishouden.recyclewaarde -= 0.1

            # invloed geslacht
            if huishouden.geslacht == "vrouw":
                huishouden.recyclewaarde += 0.05
            elif huishouden.geslacht == "man":
                huishouden.recyclewaarde -= 0.05

            # beslissing
            huishouden.recyclegedrag = huishouden.recyclewaarde > self.drempel

    # simulatiestap
    def stap(self):
        self.week += 4

        # huishoudens produceren textielafval
        for huishouden in self.huishoudens:
            huishouden.opgespaard_textielafval += huishouden.textielafval_per_tijdstap

        # recyclegedrag opnieuw berekenen
        self.bereken_recyclegedrag()

        # textiel weggooien elke 20 weken
        if self.week % 20 == 0:
            for huishouden in self.huishoudens:
                if huishouden.recyclegedrag:
                    container, afstand = self.dichtstbijzijnde_container(huishouden)

                    if container.vulgraad + huishouden.opgespaard_textielafval <= container.capaciteit:
                        container.vulgraad += huishouden.opgespaard_textielafval
                    else:
                        container.vulgraad = container.capaciteit

                huishouden.opgespaard_textielafval = 0

        # containers legen elke 8 weken
        if self.week % 8 == 0:
            for container in self.containers:
                container.ledigen()

    # resultaten weergeven
    def resultaten(self):
        print("\nRESULTATEN:\n")

        print(f"{'Huishouden':<12} {'Positie':<12} {'Buren':<6} {'Opleiding':<10} {'Inkomen':<10} {'Geslacht':<10} {'Afstand':<10} {'Opgespaard':<12} {'Recyclewaarde':<16} {'Recyclegedrag':<10}")
        print("-" * 130)

        for i, huishouden in enumerate(self.huishoudens, start=1):
            print(
                f"H{i:<11} "
                f"{str(huishouden.pos):<12} "
                f"{huishouden.sociale_gevoeligheid:<6} "
                f"{huishouden.opleidingsniveau:<10} "
                f"{huishouden.inkomen:<10} "
                f"{huishouden.geslacht:<10} "
                f"{huishouden.afstand:<10.2f} "
                f"{huishouden.opgespaard_textielafval:<12.2f} "
                f"{huishouden.recyclewaarde:<16.2f} "
                f"{str(huishouden.recyclegedrag):<10}"
            )

    # visualisatie
    def visualiseer(self):
        plt.figure(figsize=(7, 7))

        for i, h in enumerate(self.huishoudens, start=1):
            x, y = h.pos
            kleur = "blue" if h.recyclegedrag else "red"
            plt.scatter(x, y, color=kleur, s=60)
            plt.text(x, y, f"H{i}", fontsize=5, ha="center", va="center")

        for i, c in enumerate(self.containers, start=1):
            x, y = c.pos
            plt.scatter(x, y, color="green", s=100, marker="s")
            plt.text(x, y, f"C{i}", fontsize=6, ha="center", va="center")

        plt.grid()
        plt.title("Stad met huishoudens en textielcontainers")
        plt.xlim(-0.5, self.grid.width - 0.5)
        plt.ylim(-0.5, self.grid.height - 0.5)
        plt.xticks(range(self.grid.width))
        plt.yticks(range(self.grid.height))
        plt.show()


# model
model = Stad(width=10, height=10, num_people=90, people_per_container=10)

# simulatie
for week in range(4):
    print(f"\nWEEK {week + 1}")
    model.stap()
    model.visualiseer()
    model.resultaten()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# model aanmaken
model = Stad(width=10, height=10, num_people=90, people_per_container=10)

# één figuur en as maken
fig, ax = plt.subplots(figsize=(7, 7))

def update(frame):
    ax.clear()

    # 1 stap vooruit in de tijd
    model.stap()

    # huishoudens tekenen
    for i, h in enumerate(model.huishoudens, start=1):
        x, y = h.pos
        kleur = "blue" if h.recyclegedrag else "red"
        ax.scatter(x, y, color=kleur, s=60)
        ax.text(x, y, f"H{i}", fontsize=5, ha="center", va="center")

    # containers tekenen
    for i, c in enumerate(model.containers, start=1):
        x, y = c.pos
        ax.scatter(x, y, color="green", s=100, marker="s")
        ax.text(x, y, f"C{i}", fontsize=6, ha="center", va="center")

    # opmaak
    ax.set_title(f"Stad met huishoudens en textielcontainers – week {model.week}")
    ax.set_xlim(-0.5, model.grid.width - 0.5)
    ax.set_ylim(-0.5, model.grid.height - 0.5)
    ax.set_xticks(range(model.grid.width))
    ax.set_yticks(range(model.grid.height))
    ax.grid(True)
    ax.set_aspect("equal")

# animatie maken: 20 frames = 20 tijdstappen
anim = FuncAnimation(fig, update, frames=20, interval=800, repeat=False)

plt.show()

In [ ]:
import random
import math
from mesa import Model, Agent
from mesa.space import MultiGrid
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation


# Agent: huishouden
class Huishouden(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)
        self.recyclewaarde = 0
        self.sociale_gevoeligheid = 0
        self.recyclegedrag = False

        self.opleidingsniveau = random.choice(["laag", "midden", "hoog"])
        self.inkomen = random.choice(["laag", "midden", "hoog"])
        self.geslacht = random.choice(["man", "vrouw"])

        # Basis textielafval 12 kg/persoon/jaar
        self.textielafval_per_week = 0.23 * 2.10  # kg/huishouden/week
        self.textielafval_per_tijdstap = self.textielafval_per_week * 4
        self.opgespaard_textielafval = 0
        self.weggooifrequentie = 20  # in weken

        # Correctie hoeveelheid textielafval obv inkomen
        if self.inkomen == "laag":
            self.textielafval_per_tijdstap *= 0.8
        elif self.inkomen == "hoog":
            self.textielafval_per_tijdstap *= 1.2

        # Correctie hoeveelheid textielafval obv geslacht
        if self.geslacht == "man":
            self.textielafval_per_tijdstap *= 0.9
        elif self.geslacht == "vrouw":
            self.textielafval_per_tijdstap *= 1.1


# Agent: textielcontainer
class Container(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)

        self.capaciteit = 1250          # in kg (5 m³ * 250 kg/m³)
        self.vulgraad = 0
        self.ledigingsfrequentie = 8    # 1 keer per 8 weken
        self.week = 0

    def ledigen(self):
        self.vulgraad = 0


# Model: stad
class Stad(Model):
    def __init__(self, width=10, height=10, num_people=90, people_per_container=10):
        super().__init__()

        self.grid = MultiGrid(width, height, torus=False)

        self.huishoudens = []
        self.containers = []

        self.id = 0
        self.week = 0
        self.drempel = 0.5

        # aantal containers berekenen
        num_containers = math.ceil(num_people / people_per_container)

        # check of alles op het grid past
        totaal_agents = num_people + num_containers
        totaal_cellen = width * height
        if totaal_agents > totaal_cellen:
            raise ValueError(
                f"Te veel agents voor het grid: {totaal_agents} agents voor {totaal_cellen} cellen."
            )

        # alle gehele coördinaten op het grid
        beschikbare_posities = [(x, y) for x in range(width) for y in range(height)]
        random.shuffle(beschikbare_posities)

        # containers plaatsen
        for _ in range(num_containers):
            c = Container(self.next_id(), self)
            pos = beschikbare_posities.pop()
            self.grid.place_agent(c, pos)
            self.containers.append(c)

        # huishoudens plaatsen
        for _ in range(num_people):
            h = Huishouden(self.next_id(), self)
            pos = beschikbare_posities.pop()
            self.grid.place_agent(h, pos)
            self.huishoudens.append(h)

        # eerste berekening
        self.bereken_recyclegedrag()

    def next_id(self):
        self.id += 1
        return self.id

    # Manhattan-afstand op het grid
    def manhattan_afstand(self, pos1, pos2):
        return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])

    # dichtstbijzijnde container
    def dichtstbijzijnde_container(self, huishouden):
        beste_container = None
        kleinste_afstand = float("inf")

        for container in self.containers:
            afstand = self.manhattan_afstand(huishouden.pos, container.pos)

            if afstand < kleinste_afstand:
                kleinste_afstand = afstand
                beste_container = container

        return beste_container, kleinste_afstand

    # buurinvloed berekenen:
    # +0.1 voor buur die recycelt
    # -0.1 voor buur die niet recycelt
    def bereken_buurinvloed(self, huishouden):
        x, y = huishouden.pos
        aantal_buren = 0
        buurinvloed = 0

        buur_posities = [
            (x - 1, y),  # links
            (x + 1, y),  # rechts
            (x, y - 1),  # onder
            (x, y + 1)   # boven
        ]

        for bx, by in buur_posities:
            if 0 <= bx < self.grid.width and 0 <= by < self.grid.height:
                inhoud = self.grid.get_cell_list_contents([(bx, by)])

                for agent in inhoud:
                    if isinstance(agent, Huishouden):
                        aantal_buren += 1
                        if agent.recyclegedrag:
                            buurinvloed += 0.1
                        else:
                            buurinvloed -= 0.1

        return aantal_buren, buurinvloed

    # recyclegedrag bepalen
    def bereken_recyclegedrag(self):
        for huishouden in self.huishoudens:
            # random startwaarde
            huishouden.recyclewaarde = random.random()

            # buurinvloed
            aantal_buren, buurinvloed = self.bereken_buurinvloed(huishouden)
            huishouden.sociale_gevoeligheid = aantal_buren

            # afstand tot dichtstbijzijnde container
            container, afstand = self.dichtstbijzijnde_container(huishouden)
            huishouden.afstand = afstand

            # als huishouden direct naast een container woont -> altijd recyclen
            if afstand == 1:
                huishouden.recyclewaarde = self.drempel + 1
                huishouden.recyclegedrag = True
                continue

            # invloed buren
            huishouden.recyclewaarde += buurinvloed

            # invloed afstand
            huishouden.recyclewaarde -= afstand * 0.025

            # invloed inkomen
            if huishouden.inkomen == "hoog":
                huishouden.recyclewaarde += 0.1
            elif huishouden.inkomen == "laag":
                huishouden.recyclewaarde -= 0.1

            # invloed geslacht
            if huishouden.geslacht == "vrouw":
                huishouden.recyclewaarde += 0.05
            elif huishouden.geslacht == "man":
                huishouden.recyclewaarde -= 0.05

            # beslissing
            huishouden.recyclegedrag = huishouden.recyclewaarde > self.drempel

    # simulatiestap
    def stap(self):
        self.week += 4

        # huishoudens produceren textielafval
        for huishouden in self.huishoudens:
            huishouden.opgespaard_textielafval += huishouden.textielafval_per_tijdstap

        # recyclegedrag opnieuw berekenen
        self.bereken_recyclegedrag()

        # textiel weggooien elke 20 weken
        if self.week % 20 == 0:
            for huishouden in self.huishoudens:
                if huishouden.recyclegedrag:
                    container, afstand = self.dichtstbijzijnde_container(huishouden)

                    if container.vulgraad + huishouden.opgespaard_textielafval <= container.capaciteit:
                        container.vulgraad += huishouden.opgespaard_textielafval
                    else:
                        container.vulgraad = container.capaciteit

                huishouden.opgespaard_textielafval = 0

        # containers legen elke 8 weken
        if self.week % 8 == 0:
            for container in self.containers:
                container.ledigen()

    # resultaten weergeven
    def resultaten(self):
        print("\nRESULTATEN:\n")

        print(f"{'Huishouden':<12} {'Positie':<12} {'Buren':<6} {'Opleiding':<10} {'Inkomen':<10} {'Geslacht':<10} {'Afstand':<10} {'Opgespaard':<12} {'Recyclewaarde':<16} {'Recyclegedrag':<10}")
        print("-" * 130)

        for i, huishouden in enumerate(self.huishoudens, start=1):
            print(
                f"H{i:<11} "
                f"{str(huishouden.pos):<12} "
                f"{huishouden.sociale_gevoeligheid:<6} "
                f"{huishouden.opleidingsniveau:<10} "
                f"{huishouden.inkomen:<10} "
                f"{huishouden.geslacht:<10} "
                f"{huishouden.afstand:<10.2f} "
                f"{huishouden.opgespaard_textielafval:<12.2f} "
                f"{huishouden.recyclewaarde:<16.2f} "
                f"{str(huishouden.recyclegedrag):<10}"
            )

    # visualisatie
    def visualiseer(self, ax):
        ax.clear()

        for i, h in enumerate(self.huishoudens, start=1):
            x, y = h.pos
            kleur = "blue" if h.recyclegedrag else "red"
            ax.scatter(x, y, color=kleur, s=60)
            ax.text(x, y, f"H{i}", fontsize=5, ha="center", va="center")

        for i, c in enumerate(self.containers, start=1):
            x, y = c.pos
            ax.scatter(x, y, color="green", s=100, marker="s")
            ax.text(x, y, f"C{i}", fontsize=6, ha="center", va="center")

        ax.grid()
        ax.set_title(f"Stad met huishoudens en textielcontainers - week {self.week}")
        ax.set_xlim(-0.5, self.grid.width - 0.5)
        ax.set_ylim(-0.5, self.grid.height - 0.5)
        ax.set_xticks(range(self.grid.width))
        ax.set_yticks(range(self.grid.height))
        ax.set_aspect("equal")


# model
model = Stad(width=10, height=10, num_people=90, people_per_container=10)

# figuur maken
fig, ax = plt.subplots(figsize=(7, 7))

# updatefunctie voor animatie
def update(frame):
    print(f"\nWEEK {frame + 1}")
    model.stap()
    model.visualiseer(ax)
    model.resultaten()

# animatie
anim = FuncAnimation(fig, update, frames=4, interval=800, repeat=False)

plt.show()




In [ ]:
import random
import math
from mesa import Model, Agent
from mesa.space import MultiGrid
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation


# Agent: huishouden
class Huishouden(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)
        self.recyclewaarde = 0
        self.sociale_gevoeligheid = 0
        self.recyclegedrag = False

        self.opleidingsniveau = random.choice(["laag", "midden", "hoog"])
        self.inkomen = random.choice(["laag", "midden", "hoog"])
        self.geslacht = random.choice(["man", "vrouw"])

        self.textielafval_per_week = 0.23 * 2.10
        self.textielafval_per_tijdstap = self.textielafval_per_week * 4
        self.opgespaard_textielafval = 0
        self.weggooifrequentie = 20

        if self.inkomen == "laag":
            self.textielafval_per_tijdstap *= 0.8
        elif self.inkomen == "hoog":
            self.textielafval_per_tijdstap *= 1.2

        if self.geslacht == "man":
            self.textielafval_per_tijdstap *= 0.9
        elif self.geslacht == "vrouw":
            self.textielafval_per_tijdstap *= 1.1


class Container(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)
        self.capaciteit = 1250
        self.vulgraad = 0
        self.ledigingsfrequentie = 8
        self.week = 0

    def ledigen(self):
        self.vulgraad = 0


class Stad(Model):
    def __init__(self, width=10, height=10, num_people=90, people_per_container=10):
        super().__init__()

        self.grid = MultiGrid(width, height, torus=False)
        self.huishoudens = []
        self.containers = []
        self.id = 0
        self.week = 0
        self.drempel = 0.5

        num_containers = math.ceil(num_people / people_per_container)

        totaal_agents = num_people + num_containers
        totaal_cellen = width * height
        if totaal_agents > totaal_cellen:
            raise ValueError(
                f"Te veel agents voor het grid: {totaal_agents} agents voor {totaal_cellen} cellen."
            )

        beschikbare_posities = [(x, y) for x in range(width) for y in range(height)]
        random.shuffle(beschikbare_posities)

        for _ in range(num_containers):
            c = Container(self.next_id(), self)
            pos = beschikbare_posities.pop()
            self.grid.place_agent(c, pos)
            self.containers.append(c)

        for _ in range(num_people):
            h = Huishouden(self.next_id(), self)
            pos = beschikbare_posities.pop()
            self.grid.place_agent(h, pos)
            self.huishoudens.append(h)

        self.bereken_recyclegedrag()

    def next_id(self):
        self.id += 1
        return self.id

    def manhattan_afstand(self, pos1, pos2):
        return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])

    def dichtstbijzijnde_container(self, huishouden):
        beste_container = None
        kleinste_afstand = float("inf")

        for container in self.containers:
            afstand = self.manhattan_afstand(huishouden.pos, container.pos)
            if afstand < kleinste_afstand:
                kleinste_afstand = afstand
                beste_container = container

        return beste_container, kleinste_afstand

    def bereken_buurinvloed(self, huishouden):
        x, y = huishouden.pos
        aantal_buren = 0
        buurinvloed = 0

        buur_posities = [
            (x - 1, y),
            (x + 1, y),
            (x, y - 1),
            (x, y + 1)
        ]

        for bx, by in buur_posities:
            if 0 <= bx < self.grid.width and 0 <= by < self.grid.height:
                inhoud = self.grid.get_cell_list_contents([(bx, by)])

                for agent in inhoud:
                    if isinstance(agent, Huishouden):
                        aantal_buren += 1
                        if agent.recyclegedrag:
                            buurinvloed += 0.1
                        else:
                            buurinvloed -= 0.1

        return aantal_buren, buurinvloed

    def bereken_recyclegedrag(self):
        for huishouden in self.huishoudens:
            huishouden.recyclewaarde = random.random()

            aantal_buren, buurinvloed = self.bereken_buurinvloed(huishouden)
            huishouden.sociale_gevoeligheid = aantal_buren

            container, afstand = self.dichtstbijzijnde_container(huishouden)
            huishouden.afstand = afstand

            if afstand == 1:
                huishouden.recyclewaarde = self.drempel + 1
                huishouden.recyclegedrag = True
                continue

            huishouden.recyclewaarde += buurinvloed
            huishouden.recyclewaarde -= afstand * 0.025

            if huishouden.inkomen == "hoog":
                huishouden.recyclewaarde += 0.1
            elif huishouden.inkomen == "laag":
                huishouden.recyclewaarde -= 0.1

            if huishouden.geslacht == "vrouw":
                huishouden.recyclewaarde += 0.05
            elif huishouden.geslacht == "man":
                huishouden.recyclewaarde -= 0.05

            huishouden.recyclegedrag = huishouden.recyclewaarde > self.drempel

    def stap(self):
        self.week += 4

        for huishouden in self.huishoudens:
            huishouden.opgespaard_textielafval += huishouden.textielafval_per_tijdstap

        self.bereken_recyclegedrag()

        if self.week % 20 == 0:
            for huishouden in self.huishoudens:
                if huishouden.recyclegedrag:
                    container, afstand = self.dichtstbijzijnde_container(huishouden)

                    if container.vulgraad + huishouden.opgespaard_textielafval <= container.capaciteit:
                        container.vulgraad += huishouden.opgespaard_textielafval
                    else:
                        container.vulgraad = container.capaciteit

                huishouden.opgespaard_textielafval = 0

        if self.week % 8 == 0:
            for container in self.containers:
                container.ledigen()

    def visualiseer(self, ax):
        ax.clear()

        for i, h in enumerate(self.huishoudens, start=1):
            x, y = h.pos
            kleur = "blue" if h.recyclegedrag else "red"
            grootte = 90 if h.recyclegedrag else 40
            ax.scatter(x, y, s=grootte, color=kleur)
            ax.text(x, y, f"H{i}", fontsize=5, ha="center", va="center")

        for i, c in enumerate(self.containers, start=1):
            x, y = c.pos
            ax.scatter(x, y, s=120, color="green", marker="s")
            ax.text(x, y, f"C{i}", fontsize=6, ha="center", va="center")

        aantal_recyclers = sum(h.recyclegedrag for h in self.huishoudens)

        ax.set_title(f"Week {self.week} | Recyclers: {aantal_recyclers}/{len(self.huishoudens)}")
        ax.set_xlim(-0.5, self.grid.width - 0.5)
        ax.set_ylim(-0.5, self.grid.height - 0.5)
        ax.set_xticks(range(self.grid.width))
        ax.set_yticks(range(self.grid.height))
        ax.grid(True)
        ax.set_aspect("equal")


model = Stad(width=10, height=10, num_people=90, people_per_container=10)

fig, ax = plt.subplots(figsize=(7, 7))

model.visualiseer(ax)

def update(frame):
    model.stap()
    model.visualiseer(ax)

anim = FuncAnimation(
    fig,
    update,
    frames=20,
    interval=1000,
    repeat=False,
    cache_frame_data=False
)

from IPython.display import HTML
HTML(anim.to_jshtml())

